# Compare offline and online-binned mixed layer tracer budgets

This notebook contains code to compare the online method (most accurate) with offline methods based on monthly and daily binning of heat budget diagnostics into the mixed layer.

In [ ]:
#Load required packages
%matplotlib inline
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
import cftime
from tqdm import tqdm

import cmocean as cm
import sys, os
import datetime

from dask.distributed import Client

In [ ]:
# Load workers:
client = Client(n_workers=4)
client

In [ ]:
# change directory to Figures/ subfolder for saving images
os.chdir('access-om2-analysis/access-om2-sst-budget/Figures')

# Load data

### Define paths, region to analyse and time period to analyse

In [ ]:
base = '/scratch/e14/rmh561/access-om2/archive/025deg_jra55_iaf_cycle6_online_mlt/'
output = 364 # 364 = 2017
#output = 365 # 365 = 2018
#output = 366 # 366 = 2019 - contains 3D daily budget diagnostics for quantifying correlation errors

tmp_folder = base + 'post_processed_diags/'

base2 = base + 'output%03d/ocean/' % output

# Climatology:
clim_str = 'output336-365' # 336-365 = 1989-2018
clim_label = '1989-2018'

# Subsample regions:
#reg = [-100, 20, 0, 70] # North Atlantic
#reg = [-100, -40, 0, 30] # North Atlantic
#reg = [-230, -190, -50, -10] # EAC
reg = [135-360,175-360, -60, -20] # SE Aus (Kajtar et al. 2022)
#reg = [None,None,None,None] # Globe
#reg = [-270, -70, -60, 60] # Pacific
#reg = [-270, -210, -20, 20] # Maritime continent
#reg = [-270, -180, -45, 0] # Australia

# Subsample time:
#times = slice('2019-01-01','2019-01-31')#lice(None,None)
#times_snap = slice('2019-01-01','2019-02-01') # Note; this must be 1 more than times.
times = slice('2017-09-01',None)
times_snap = slice('2017-09-01',None) # Note; this must be 1 more than times.
#times = slice(None,None)
#times_snap = slice(None,None) # Note; this must be 1 more than times.

chunks2D = {'time':1,'yt_ocean':216,'xt_ocean':240}
chunks3D = {'time':1,'st_ocean':25,'yt_ocean':324,'xt_ocean':360}

### Load grid and set constants

In [ ]:
ds_grid = xr.open_dataset(base2 + 'ocean_grid.nc',chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))#.isel(time=times)
rho0 = 1035.
Cp = 3992.10322329649

### Load daily data

#### Standard variables:

In [ ]:
ds_day = xr.open_dataset(base2 + 'ocean_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_day = ds_day.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day.time.values]})
ds_day.average_DT.data = ds_day.average_DT*np.timedelta64(1,'D')
ds_day = ds_day.sel(time=times)

#### Pre-computed MLT budget

In [ ]:
# Pre-computed standard average online daily
mlt_budget_stavg_daily = xr.open_dataset(tmp_folder + 'mlt_budget_stavg_daily_online_output%03d.nc' % output).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).sel(time=times)

In [ ]:
# Pre-computed standard average online climatology (1989-2018, outputs 336-365):
mlt_budget_stavg_clim = xr.open_dataset(tmp_folder + 'mlt_budget_stavg_daily_online_' + clim_str + '_monthly_mean.ncea.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

### 3D budget data for offline budgets

In [ ]:
# # monthly data:
ds_mon_budget_3d = xr.open_dataset(base2 + 'ocean_budget_month_3d.nc',decode_times=False,chunks=chunks3D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_day_budget_3d = xr.open_dataset(base2 + 'ocean_budget_daily_3d.nc',decode_times=False,chunks=chunks3D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_mon = xr.open_dataset(base2 + 'ocean_month.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

# Snapshots for standard average tendency computation:
ds_mon_snapshot = xr.open_dataset(base2 + 'ocean_snapshot_month.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
# Add previous output for last element:
ds_mon_snapshot_m1 = xr.open_dataset(base2.replace(str(output),str(output-1)) + 'ocean_snapshot_month.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_mon_snapshot = xr.concat([ds_mon_snapshot_m1.isel(time=-1),ds_mon_snapshot],dim='time')

# Fix time variable by decoding time by hand (see https://forum.access-hive.org.au/t/cftime-vs-datetime64-time-encoding-issues-with-access-om2-025-omip-2-run/4085);
ds_mon_budget_3d = ds_mon_budget_3d.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_mon_budget_3d.time.values]})
ds_day_budget_3d = ds_day_budget_3d.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_budget_3d.time.values]})
ds_mon = ds_mon.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_mon.time.values]})
ds_mon_snapshot = ds_mon_snapshot.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_mon_snapshot.time.values]})

# Fix average_DT by decoding by hand:
ds_mon_budget_3d.average_DT.data = ds_mon_budget_3d.average_DT*np.timedelta64(1,'D')
ds_day_budget_3d.average_DT.data = ds_day_budget_3d.average_DT*np.timedelta64(1,'D')
ds_mon.average_DT.data = ds_mon.average_DT*np.timedelta64(1,'D')

# Subselect time period:
ds_mon_budget_3d = ds_mon_budget_3d.sel(time=times)
ds_day_budget_3d = ds_day_budget_3d.sel(time=times)
ds_mon = ds_mon.sel(time=times)
ds_mon_snapshot = ds_mon_snapshot.sel(time=times_snap)

In [ ]:
# Snapshots for standard average tendency computation:
ds_day_snapshot = xr.open_dataset(base2 + 'ocean_snapshot_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
# Add previous output for last element:
ds_day_snapshot_m1 = xr.open_dataset(base2.replace(str(output),str(output-1)) + 'ocean_snapshot_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_day_snapshot = xr.concat([ds_day_snapshot_m1.isel(time=-1),ds_day_snapshot],dim='time')

# Fix time variable by decoding time by hand (see https://forum.access-hive.org.au/t/cftime-vs-datetime64-time-encoding-issues-with-access-om2-025-omip-2-run/4085);
ds_day_snapshot = ds_day_snapshot.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_snapshot.time.values]})

# Fix average_DT by decoding by hand:
ds_day_snapshot = ds_day_snapshot.sel(time=times_snap)

# Compare offline and daily budgets

What diagnostics are needed to do the offline binning:
- Closed 3D heat budget
- `dzt` and `mld` time-averages to bin the 3D diagnostics into the mixed layer.
- Snapshots of `temp_in_mld` for tendency calculation.

## Define function to compute offline mixed layer temperature budget

In [ ]:
def compute_mixed_layer_temperature_budget_offline(ds_budget,ds_budget_2d,mld,dzt):
    """
    Compute mixed layer temperature budget

    Inputs:
    ds_budget -> dataset containing time-averaged 3D budget quantities
    ds_budget_2d -> dataset containing time-averaged 2D (surface layer only) budget quantities (these are just added in without summing)
    mld -> dataarray containing time-averaged mixed layer depth
    dzt -> datarray containing time-averaged grid cell thicknesses
    temp_mld_snap -> datarray containing snapshots of temp*rho0*dzt summed over mld
    mld_snap -> dataarray containing snapshots of mld

    Outputs:
    ds_budget_in_mld -> dataset containing all the "internal" (i.e. not including mlt tendency and entrainment) terms in the diagnosed mixed layer temperature budget (2D), in units of degC/second
    """

    # Sum over mixed layer:
    ds_budget_in_mld = ds_budget.isel(st_ocean=0,drop=True).copy(deep=True)
    for var in list(ds_budget_2d.data_vars):
        ds_budget_in_mld[var] = xr.zeros_like(ds_budget['fixedh_tendency']).isel(st_ocean=0,drop=True).copy(deep=True)
    for ti in range(len(ds_budget.time)): # Loop over time
        dzt_ti = dzt.isel(time=ti).load()       
        dzt_ti_bot = dzt_ti.cumsum('st_ocean')  # Depth (from free surface) of bottom of each cell
        ds_budget_ti = ds_budget.isel(time=ti).load()
        mld_ti = mld.isel(time=ti).load()
        for var in list(ds_budget.data_vars):
            ds_budget_in_mld[var][ti,:,:] = ds_budget_ti[var].where(dzt_ti_bot<mld_ti).sum('st_ocean')   # Include all of cells that lie completely in the mixed layer
            for k in range(len(ds_budget_ti.st_ocean)-1):
                ds_budget_in_mld[var][ti,:,:] += xr.where(np.logical_and(dzt_ti_bot[k,:,:]>mld_ti,dzt_ti_bot[k+1,:,:]<mld_ti),((mld_ti-dzt_ti_bot[k,:,:])/dzt_ti[k,:,:])*ds_budget_ti[var][k,:,:],0.) # Include only a fraction of cells that lie partially within the mixed layer
            ds_budget_in_mld[var][ti,:,:] = ds_budget_in_mld[var][ti,:,:]/rho0/Cp/mld_ti # Convert units from Wm-2 to degC/sec

        # Add 2D surface layer variables:
        surf_frac = xr.where(dzt_ti_bot[0,:,:]>mld_ti,mld_ti/dzt_ti[0,:,:],1.)            # In regions where the mixed layer depth is shallower than the thickness of the surface grid cell, take only that fraction from the 2D variables
        for var in list(ds_budget_2d.data_vars):
            ds_budget_in_mld[var][ti,:,:] = (surf_frac*ds_budget_2d[var][ti,:,:]/rho0/Cp/mld_ti).load()

    # Compute residual for check:
    ds_budget_in_mld['residual'] = ds_budget_in_mld['fixedh_tendency'] - ds_budget_in_mld[list(ds_budget_in_mld.data_vars)[1:]].to_array().sum('variable')
        
    return(ds_budget_in_mld)

## Compute daily and monthly offline binned mixed layer temperature budgets (standard averaging):

In [ ]:
# Offline budget terms grouping:
bud_var_grps = {'advection':['temp_advection','temp_submeso','temp_vdiffuse_k33','neutral_diffusion_temp','neutral_gm_temp'],
                'vert_mixing':['temp_vdiffuse_diff_cbt','temp_nonlocal_KPP'],
                'surface_flux':['temp_vdiffuse_sbc','frazil_3d','temp_rivermix'],
                'sw_pen':['sw_heat']}
bud_2d_vars = ['temp_eta_smooth','sfc_hflux_pme']

In [ ]:
# Compute monthly budget, by month:

# Add sfc_hflux_pme from standard monthly diagnostics file:
ds_mon_budget_3d['sfc_hflux_pme'] = ds_mon['sfc_hflux_pme']

mlt_budget_stavg_monthly_offline_uncat = []

# Loop over month:
for ti in tqdm(range(len(ds_mon_budget_3d.time))):
    
    # Group terms:
    ds_mon_budget_3d_reduced = ds_mon_budget_3d.isel(time=slice(ti,ti+1))['temp_tendency'].rename('fixedh_tendency').to_dataset().copy(deep=True)
    for var in bud_var_grps.keys():
        ds_mon_budget_3d_reduced[var] = ds_mon_budget_3d.isel(time=slice(ti,ti+1))[bud_var_grps[var][0]].load()
        if (len(bud_var_grps[var])>1):
            for raw_var in bud_var_grps[var][1:]:
                ds_mon_budget_3d_reduced[var] += ds_mon_budget_3d.isel(time=slice(ti,ti+1))[raw_var].load()

    ds_mon_budget_3d_reduced_2d = ds_mon_budget_3d.isel(time=slice(ti,ti+1))[bud_2d_vars].load()

    # Do monthly computation:
    bud = compute_mixed_layer_temperature_budget_offline(ds_mon_budget_3d_reduced,ds_mon_budget_3d_reduced_2d,ds_mon.mld.isel(time=slice(ti,ti+1)),ds_mon.dzt.isel(time=slice(ti,ti+1)))
    # Add 2D vars to sbc term:
    for var in bud_2d_vars:
        bud['surface_flux'] += bud[var]
        bud = bud.drop_vars([var])

    bud = compute_tendency_entrainment(bud,ds_mon_snapshot.isel(time=slice(ti,ti+2)).temp_in_mld/rho0)

    mlt_budget_stavg_monthly_offline_uncat.append(bud)

mlt_budget_stavg_monthly_offline = xr.concat(mlt_budget_stavg_monthly_offline_uncat,dim='time')

In [ ]:
# Save to file:
mlt_budget_stavg_monthly_offline.to_netcdf(tmp_folder + 'mlt_budget_stavg_monthly_offline_2023.nc')

In [ ]:
%%time
# Compute daily budget, in blocks:
mlt_budget_stavg_daily_offline_uncat = []

bs = 5; tl = len(ds_day_budget_3d.time)
blocks = [range(tl)[x*bs:(x+1)*bs] for x in range(int(np.ceil(tl/bs)))]
blocks_snap = [range(tl+1)[x*bs:(x+1)*bs + 1] for x in range(int(np.ceil(tl/bs)))]

for i in tqdm(np.arange(55,len(blocks)+1)):#range(len(blocks))):
    ds_day_budget_3d_reduced = ds_day_budget_3d.isel(time=blocks[i])['temp_tendency'].rename('fixedh_tendency').to_dataset()
    
    for var in bud_var_grps.keys():
        ds_day_budget_3d_reduced[var] = ds_day_budget_3d.isel(time=blocks[i])[bud_var_grps[var][0]]
        if (len(bud_var_grps[var])>1):
            for raw_var in bud_var_grps[var][1:]:
                ds_day_budget_3d_reduced[var] += ds_day_budget_3d.isel(time=blocks[i])[raw_var]
                
    ds_day_budget_3d_reduced_2d = ds_day_budget_3d.isel(time=blocks[i])[bud_2d_vars]

    ds_day_budget_3d_reduced.load()
    ds_day_budget_3d_reduced_2d.load()
    mld = ds_day.mld.isel(time=blocks[i]).load()
    dzt = ds_day_budget_3d.dzt.isel(time=blocks[i]).load()
    
    bud = compute_mixed_layer_temperature_budget_offline(ds_day_budget_3d_reduced,ds_day_budget_3d_reduced_2d,mld,dzt)
    
    for var in bud_2d_vars:
        bud['surface_flux'] += bud[var]
        bud = bud.drop_vars([var])

    temp_in_mld = (ds_day_snapshot.isel(time=blocks_snap[i]).temp_in_mld/rho0).load()
    bud = compute_tendency_entrainment(bud,temp_in_mld)

    #    mlt_budget_stavg_daily_offline_uncat.append(bud)
    bud.to_netcdf(tmp_folder + 'mlt_budget_stavg_daily_offline_2023_block%03d.nc' % i)

In [ ]:
# Concat and save to a single file:
ds_cat = []

for i in tqdm(range(73)):
    ds = xr.open_dataset(tmp_folder + 'mlt_budget_stavg_daily_offline_2023_block%03d.nc' % i).load()
    ds_cat.append(ds)
ds = xr.concat(ds_cat,dim='time')

ds.to_netcdf(tmp_folder + 'mlt_budget_stavg_daily_offline_2023.nc')

In [ ]:
# Create monthly means from daily files and save back to file:
fname = '/g/data/e14/rmh561/mlt_budget_temporary/mlt_budget_stavg_daily_online_output365.nc'
ds  = xr.open_dataset(fname, chunks={'time':61, 'yt_ocean': 180, 'xt_ocean': 240})
ds_mon = xr.zeros_like(ds.resample(time='1ME').mean()).copy(deep=True)

for var in tqdm(ds.data_vars):
    ds_mon[var] = ds[var].resample(time='1ME').mean().load()
ds_mon = ds_mon.assign_coords({'time':ds.time.resample(time='1ME').mean()})
ds_mon.to_netcdf(fname[:-3] + '_monthly_mean.nc')

#### Spatial plots comparing monthly, daily offline and daily:

In [ ]:
# Load from file:
mlt_budget_stavg_monthly_offline = xr.open_dataset('/g/data/e14/rmh561/mlt_budget_temporary/mlt_budget_stavg_monthly_offline_output370.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).load()
mlt_budget_stavg_daily_offline  = xr.open_dataset('/g/data/e14/rmh561/mlt_budget_temporary/mlt_budget_stavg_daily_offline_output370_monthly_mean.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).load()
mlt_budget_stavg_daily_online  = xr.open_dataset('/g/data/e14/rmh561/mlt_budget_temporary/mlt_budget_stavg_daily_online_output370_monthly_mean.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).load()

In [ ]:
regs = {'Warm Pool':[-200, -160, -10, 10],
        'Tasman Sea':[-205, -190, -40, -25],
        'Cold Tongue':[-150,-90,-5,5],
       }

In [ ]:
# Region selection:
fig = plt.figure(figsize=(30,15))
mlt_budget_stavg_monthly_offline['vert_mixing'].mean('time').plot()
for key in regs.keys():
    sreg = regs[key]
    plt.plot([sreg[0],sreg[1],sreg[1],sreg[0],sreg[0]],[sreg[2],sreg[2],sreg[3],sreg[3],sreg[2]],'-k')
plt.gca().set_facecolor('k')

In [ ]:
# Spatial plots:
fig, axs = plt.subplots(nrows=3, ncols=6, figsize=(15,6),layout='constrained')

vars = ['mlt_tendency','entrainment','advection','vert_mixing','surface_flux','sw_pen']
labels = ['Tendency','Entrainment','Advection','Vertical Mixing','Surface fluxes','SW penetration']
budgets = [mlt_budget_stavg_monthly_offline.mean('time'),
           mlt_budget_stavg_daily_offline.mean('time'),
           mlt_budget_stavg_daily_online.mean('time'),
           #mlt_budget_stavg_daily_online.mean('time') - mlt_budget_stavg_monthly_offline.mean('time'),
           #mlt_budget_stavg_daily_online.mean('time') - mlt_budget_stavg_daily_offline.mean('time')
          ]
budget_labels = ['Monthly Offline','Daily Offline','Online','Online - Monthly Offline','Online - Daily Offline']
unit_conv = 86400*30.5
clims = [1,1,1,2.5,5,5]

for i in range(len(budgets)):
    for j, var in enumerate(vars):
        if i == 0:
            (budgets[i][var]*unit_conv).plot(ax=axs[i][j],cmap='RdBu_r',vmin=-clims[j],vmax=clims[j],extend='both',cbar_kwargs={'label':labels[j] + ' ($^\circ$C/month)','shrink':0.7,'location':'top'})
        else:
            (budgets[i][var]*unit_conv).plot(ax=axs[i][j],cmap='RdBu_r',vmin=-clims[j],vmax=clims[j],extend='both',add_colorbar=False)
        axs[i][j].set_ylabel('')
    axs[i][0].set_ylabel(budget_labels[i])
    
for key in regs.keys():
    sreg = regs[key]
    axs[2][0].plot([sreg[0],sreg[1],sreg[1],sreg[0],sreg[0]],[sreg[2],sreg[2],sreg[3],sreg[3],sreg[2]],'-k',linewidth=0.5)

for ax in axs.reshape(-1):
    ax.set_ylim([-60,60])
    ax.set_xlabel('')
    ax.set_title('')
    ax.set_facecolor('k')
    ax.set_xticklabels([])
    ax.set_yticklabels([])

plt.savefig('MLT_budget_2023_Online_Offline_Comparison_Pacific.png',dpi=300)

In [ ]:
# Compute spatial averages:

budgets = {}

def area_average(budget,reg):

    total_area = ds_grid.area_t.sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).mean(['xt_ocean','yt_ocean'])
    budget_av = (budget*ds_grid.area_t).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).mean(['xt_ocean','yt_ocean'])/total_area
    return(budget_av)

for key in tqdm(regs.keys()):
    sreg = regs[key]
    budgets[key] = [area_average(mlt_budget_stavg_monthly_offline,sreg).load(),
               area_average(mlt_budget_stavg_daily_offline,sreg).load(),
               area_average(mlt_budget_stavg_daily_online,sreg).load()
              ]

In [ ]:
# Time series across specific regions:
fig, axs = plt.subplots(nrows=len(regs.keys()), ncols=1, figsize=(12,12))

vars = ['mlt_tendency','entrainment','advection','vert_mixing','surface_flux','sw_pen']
labels = ['Tendency','Entrainment','Advection','Vertical Mixing','Surface fluxes','SW penetration']
cols = ['k','r','b','g','m','c']
typs= ['-','--',':']
budget_labels = ['Monthly Offline','Daily Offline','Online']
unit_conv = 86400*30.5

for k, key in enumerate(regs.keys()):
    sreg = regs[key]
    for i in range(len(budgets)):
        for j, var in enumerate(vars):
            if j == 0 and k == 0:
                (budgets[key][i][var]*unit_conv).plot(ax=axs[k],color=cols[j],linestyle=typs[i],linewidth=2,label=budget_labels[i])
            elif i == 0 and k == 1:
                (budgets[key][i][var]*unit_conv).plot(ax=axs[k],color=cols[j],linestyle=typs[i],linewidth=2,label=labels[j])
            else:
                (budgets[key][i][var]*unit_conv).plot(ax=axs[k],color=cols[j],linestyle=typs[i],linewidth=2)
    axs[k].set_title(key)
axs[0].legend()
axs[1].legend()
plt.tight_layout()
plt.savefig('MLT_budget_2023_Online_Offline_Comparison_Time_Series.png',dpi=300)

In [ ]:

fig, axes = plt.subplots(nrows=3,ncols=1,figsize=(13, 15))

months = {'Warm Pool':range(12),
        'Tasman Sea':[9,10,11,0,1,2],
        'Cold Tongue':range(12),
       }
month_lab = {'Warm Pool':'Annual',
        'Tasman Sea':'Oct-Dec, Jan-Feb',
        'Cold Tongue':'Annual'}

# Bar plots across specific regions:
for i, key in enumerate(budgets.keys()):
    ax = axes[i]
    
    ds = xr.concat(budgets[key],dim='class').assign_coords({'class':['Monthly Offline','Daily Offline','Online']}).isel(time=months[key]).mean('time')
    
    # Add another variable:
    ds['surface_flux_total'] = ds['surface_flux']+ds['sw_pen']
    ds['vert_total'] = ds['surface_flux']+ds['sw_pen']+ds['vert_mixing']
    
    variables = ['mlt_tendency','entrainment','advection','vert_mixing','surface_flux','sw_pen','residual','surface_flux_total','vert_total']
    classes = ['Monthly Offline','Daily Offline','Online']
    labels = ['Tendency','Entrainment','Advection','Vertical Mixing','Surface fluxes','SW penetration','Residual','Surface fluxes +\nSW penetration','Surface fluxes+ \nSW penetration \n+ Vertical Mixing']
    
    # Convert to DataFrame for easier plotting
    df = ds.to_dataframe()
    
    # Reset index to get 'class' as a column
    df = df.reset_index()
    
    # Parameters
    num_vars = len(variables)
    num_classes = len(classes)
    bar_width = 0.25
    x = np.arange(num_vars)  # One x position per variable
    
    unit_conv = 86400*30.5
    # Create figure
    
    # Plot each class as a separate bar group
    for i, cls in enumerate(classes):
        # Get values for this class across all variables
        values = [df[df['class'] == cls][var].values[0]*unit_conv for var in variables]
        
        # Offset x positions for each class
        ax.bar(x + i * bar_width, values, width=bar_width, label=cls)
    
    # Formatting
    ax.set_xticks(x + bar_width)
    ax.set_xticklabels(labels)
    ax.set_ylabel("$^\circ$C/month")
    ax.set_title(key + ' Mixed Layer Temperature Budget ' +month_lab[key] + ' 2023')
    ax.legend(title="Budget")
    ax.grid()
    plt.tight_layout()
plt.savefig('MLT_budget_Online_Offline_Comparison_Bar.png',dpi=250)